# Lekcja 3 — Kanał z fadingiem

## Cel nauki
Progresja złożoności kanału (dokładnie jak w planie nauki):

1. **AWGN**: $y = x + n$
2. **Flat fading**: $y = h x + n$ (jeden współczynnik $h$)
3. **Frequency selective**: $y = H(f) \cdot x + n$ (różne $h$ na subcarrierach)
4. **TDL/CDL** — modele 3GPP (realistyczny 5G)

## Pojęcia
- **Channel response** $H$ — jak kanał zniekształca sygnał
- **Multipath** — wiele ścieżek propagacji → selectivity w częstotliwości
- **Doppler** — ruch → zmiana $h$ w czasie

## Co zastąpi sieć?
Klasyczny receiver robi **Channel Estimation** (z pilotów) + **Equalization**.
Sieć neuronowa uczy się robić to implicitnie z całego grida.


In [1]:
import sys
from pathlib import Path

# Dodaj src/ do PYTHONPATH
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import torch

try:
    import sionna as sn
    import sionna.phy
except ImportError as e:
    raise ImportError(
        "Brak Sionny. Uruchom z katalogu magisterka/: ./scripts/drun sync"
    ) from e

from src.utils.setup import print_environment, get_device

sn.phy.config.seed = 42
device = get_device()
print_environment()


=== Środowisko neural-receiver ===
Sionna:  2.0.1
PyTorch: 2.13.0+cu129
CUDA:    True
Device:  cuda
GPU:     NVIDIA GeForce GTX 1660 SUPER


## Etap 1 — AWGN (przypomnienie)

In [2]:
NUM_BITS = 2
constellation = sn.phy.mapping.Constellation("qam", NUM_BITS)
mapper = sn.phy.mapping.Mapper(constellation=constellation)
demapper = sn.phy.mapping.Demapper("app", constellation=constellation)
awgn = sn.phy.channel.AWGN()

bits = sn.phy.mapping.BinarySource()([100, 512])
x = mapper(bits)
no = sn.phy.utils.ebnodb2no(15.0, NUM_BITS, 1.0)
y = awgn(x, no)
llr = demapper(y, no)
print("BER AWGN:", float((bits != (llr > 0)).float().mean()))


BER AWGN: 0.0


## Etap 2 — Flat fading (1 współczynnik h)

In [ ]:
from sionna.phy.channel import GenerateFlatFadingChannel

# Macierz kanału: [batch, num_rx_ant, num_tx_ant]
flat_gen = GenerateFlatFadingChannel(num_tx_ant=1, num_rx_ant=1)
h = flat_gen(batch_size=x.shape[0])  # h ~ CN(0,1)
print("Shape h:", h.shape)

# y = h*x + n  (flat fading — to samo h dla wszystkich symboli w batchu)
h_scalar = h[:, 0, 0].unsqueeze(-1)  # [batch, 1]
y_fade = h_scalar * x + awgn(torch.zeros_like(x), no)
print("Średnie |y| vs |x|:", torch.abs(y_fade).mean().item(), torch.abs(x).mean().item())


Shape h: torch.Size([100, 1, 1])
Średnie |y| vs |x|: 0.9134539365768433 0.9999998807907104


## Etap 3 — Frequency selective (OFDM + TDL)

In [4]:
from sionna.phy.ofdm import ResourceGrid, ResourceGridMapper, OFDMModulator, OFDMDemodulator
from sionna.phy.channel import OFDMChannel
from sionna.phy.channel.tr38901 import TDL

rg = ResourceGrid(
    num_ofdm_symbols=14,
    fft_size=64,
    subcarrier_spacing=30e3,
    cyclic_prefix_length=6,
    pilot_pattern="kronecker",
    pilot_ofdm_symbol_indices=[2, 11],
)
rg_mapper = ResourceGridMapper(rg)
modulator = OFDMModulator(rg.cyclic_prefix_length)
demodulator = OFDMDemodulator(rg.fft_size, 0, rg.cyclic_prefix_length)

# TDL-A — profil delay spread (3GPP)
tdl = TDL(model="A", delay_spread=30e-9, carrier_frequency=3.5e9, min_speed=0.0)

NUM_BPS = 2
mapper_q = sn.phy.mapping.Mapper(constellation=sn.phy.mapping.Constellation("qam", NUM_BPS))
bits_ofdm = sn.phy.mapping.BinarySource()([8, rg.num_data_symbols * NUM_BPS])
sym = mapper_q(bits_ofdm).reshape(8, 1, 1, rg.num_data_symbols)
x_grid = rg_mapper(sym)
x_time = modulator(x_grid)

# OFDMChannel łączy TDL + AWGN w domenie częstotliwości
from sionna.phy.channel import OFDMChannel

ofdm_channel = OFDMChannel(tdl, rg, add_awgn=True, normalize_channel=True, return_channel=True)
no_t = sn.phy.utils.ebnodb2no(10.0, NUM_BPS, 1.0)
y_grid, h_freq = ofdm_channel(x_grid, no_t)

print("Shape y_grid (RX):", y_grid.shape)
print("Shape h_freq (idealna wiedza o kanale):", h_freq.shape)


Shape y_grid (RX): torch.Size([8, 1, 1, 14, 64])
Shape h_freq (idealna wiedza o kanale): torch.Size([8, 1, 1, 1, 1, 14, 64])


## Podsumowanie
- **Flat fading** — ten sam $h$ dla wszystkich subcarrierów
- **Selective fading** — $h$ zależy od częstotliwości → potrzebne piloty
- **TDL/CDL** — parametryzowane modele do treningu i testu generalizacji

## Ćwiczenie
Narysuj `|h_freq|` dla pierwszego batcha jako heatmapę (subcarrier × OFDM symbol).

**Następna lekcja:** `04_mimo_basics.ipynb`
